In [ ]:
import requests
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit
import joblib, os, json, time, logging, warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')
np.random.seed(42)

#Fetch Data

In [ ]:

# ============================================================
# fetch_data.py  —   XAU/USD
# ============================================================

#   python fetch_data
#
# بيحفظ: XAUUSD_5min_raw.csv
# ============================================================

import requests
import pandas as pd
import time
import logging
from datetime import datetime, timedelta

# ─── إعدادات ───────────────────────────────────────────────
TWELVE_DATA_API_KEY = ""
SYMBOL       = "XAU/USD"
SYMBOL_SAFE  = "XAUUSD"
CSV_RAW_FILE = f"{SYMBOL_SAFE}_5min_raw.csv"

TARGET_CANDLES = 400_000   # ← غيّري الرقم ده لو عايزي أقل
MAX_REQUESTS   = 80        # 80 × 5000 = 400k شمعة

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[logging.StreamHandler()]
)

# ─── Fetcher ────────────────────────────────────────────────
class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key):
        self.api_key = api_key

    def fetch_range(self, symbol=SYMBOL, interval="5min",
                    start_date=None, end_date=None, outputsize=5000):
        params = {
            'symbol': symbol, 'interval': interval,
            'apikey': self.api_key, 'format': 'JSON',
            'order': 'ASC', 'outputsize': outputsize
        }
        if start_date: params['start_date'] = start_date
        if end_date:   params['end_date']   = end_date
        try:
            r = requests.get(self.BASE_URL, params=params, timeout=30)
            if r.status_code != 200:
                logging.error(f"HTTP {r.status_code}")
                return pd.DataFrame()
            data = r.json()
            if 'values' not in data:
                logging.error(f"API error: {data.get('message','unknown')}")
                return pd.DataFrame()
            df = pd.DataFrame(data['values'])
            if 'volume' not in df.columns:
                df['volume'] = 0
            for col in ['open', 'high', 'low', 'close', 'volume']:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            df['datetime'] = pd.to_datetime(df['datetime'])
            df.sort_values('datetime', inplace=True)
            df.reset_index(drop=True, inplace=True)
            return df
        except Exception as e:
            logging.error(f"Fetch error: {e}")
            return pd.DataFrame()


def fetch_full_history(target_candles=TARGET_CANDLES, interval="5min"):
    fetcher    = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    all_data   = []
    total      = 0
    end_date   = datetime.utcnow()
    prev_oldest = None

    for req in range(MAX_REQUESTS):
        start_date = end_date - timedelta(seconds=5000 * 300)
        batch = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=start_date.strftime("%Y-%m-%d %H:%M:%S"),
            end_date=end_date.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=5000
        )
        if batch.empty:
            logging.warning("batch فاضي — توقف")
            break

        all_data.append(batch)
        total += len(batch)
        oldest = batch['datetime'].min()
        logging.info(f"Batch {req+1:02d}: {len(batch):,} شمعة | إجمالي: {total:,} | أقدم: {oldest.date()}")

        if total >= target_candles:
            logging.info("✅ وصلنا للعدد المطلوب")
            break
        if prev_oldest == oldest:
            logging.info("⚠️ نفس أقدم تاريخ — توقف")
            break

        prev_oldest = oldest
        end_date    = oldest - timedelta(seconds=1)
        time.sleep(0.35)   # نحترم API rate limit

    if not all_data:
        logging.error("❌ مفيش داتا اتجلبت")
        return pd.DataFrame()

    df = pd.concat(all_data, ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='first', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)

    if len(df) > target_candles:
        df = df.iloc[-target_candles:].reset_index(drop=True)

    logging.info(f"\n✅ الداتا النهائية: {len(df):,} شمعة")
    logging.info(f"   من: {df['datetime'].min().date()} → {df['datetime'].max().date()}")
    return df


def get_or_update_data(csv_path=CSV_RAW_FILE,
                       target_candles=TARGET_CANDLES,
                       force_refresh=False):
    """
    لو الـ CSV موجود → يحمّله مباشرة (لا API calls)
    لو مش موجود أو force_refresh=True → يجيب من الـ API
    """
    if not force_refresh and __import__('os').path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df['datetime'] = pd.to_datetime(df['datetime'])
        logging.info(f"✅ CSV محمّل: {len(df):,} شمعة ({csv_path})")
        return df

    logging.info(f"جلب {target_candles:,} شمعة من API...")
    df = fetch_full_history(target_candles=target_candles)
    if not df.empty:
        df.to_csv(csv_path, index=False)
        logging.info(f"✅ محفوظ: {csv_path}")
    return df


# ─── تشغيل مباشر ────────────────────────────────────────────
if __name__ == "__main__":
    import os
    if os.path.exists(CSV_RAW_FILE):
        ans = input(f"الملف '{CSV_RAW_FILE}' موجود. تجديد البيانات؟ (y/n): ").strip().lower()
        force = (ans == 'y')
    else:
        force = True

    df = get_or_update_data(force_refresh=force)
    if not df.empty:
        print(f"\n✅ جاهز! {len(df):,} شمعة محفوظة في '{CSV_RAW_FILE}'")
    else:
        print("❌ فشل جلب البيانات")


✅ جاهز! 282,619 شمعة محفوظة في 'XAUUSD_5min_raw.csv'


In [9]:
import os
print(os.listdir('/content'))

['.config', 'fetch_data.py', 'training_patterns.log', 'models', 'XAUUSD_5min_raw.csv', 'sample_data']


In [10]:
from google.colab import files
files.upload()

Saving fetch_data.py to fetch_data (1).py


{'fetch_data (1).py': b'# -*- coding: utf-8 -*-\n# ============================================================\n# fetch_data.py  \xe2\x80\x94  \xd8\xac\xd9\x84\xd8\xa8 \xd8\xaf\xd8\xa7\xd8\xaa\xd8\xa7 XAU/USD (\xd8\xb4\xd8\xba\xd9\x91\xd9\x84\xd9\x87 \xd9\x85\xd8\xb1\xd8\xa9 \xd9\x88\xd8\xa7\xd8\xad\xd8\xaf\xd8\xa9 \xd8\xa8\xd8\xb3)\n# ============================================================\n# \xd8\xa7\xd9\x84\xd8\xa7\xd8\xb3\xd8\xaa\xd8\xae\xd8\xaf\xd8\xa7\xd9\x85:\n#   python fetch_data.py\n#\n# \xd8\xa8\xd9\x8a\xd8\xad\xd9\x81\xd8\xb8: XAUUSD_5min_raw.csv\n# ============================================================\n\nimport requests\nimport pandas as pd\nimport time\nimport logging\nfrom datetime import datetime, timedelta\n\n# \xe2\x94\x80\xe2\x94\x80\xe2\x94\x80 \xd8\xa5\xd8\xb9\xd8\xaf\xd8\xa7\xd8\xaf\xd8\xa7\xd8\xaa \xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x8

In [ ]:
from google.colab import files
files.upload()

Saving train_patterns_v3_classical.py to train_patterns_v3_classical.py


{'train_patterns_v3_classical.py': b'# -*- coding: utf-8 -*-\n# ============================================================\n# train_patterns.py  v3.3  \xe2\x80\x94  Classical Pattern Recognition\n# ============================================================\n# \xd8\xa7\xd9\x84\xd9\x85\xd8\xaf\xd8\xa7\xd8\xb1\xd8\xb3 \xd8\xa7\xd9\x84\xd9\x85\xd8\xaf\xd8\xb9\xd9\x88\xd9\x85\xd8\xa9:\n#   Classic  : H&S, Double, Triple, Triangle, Flag, Wedge,\n#              Channel, Cup & Handle, Rectangle\n#   Harmonic : Gartley, Butterfly, Bat, Crab, Shark, Cypher + PRZ\n#   Elliott  : Impulse (Strict Rules), Corrective A-B-C,\n#              Extended W3, Diagonal, Alternation Rule\n#\n# v3.3 \xe2\x80\x94 \xd8\xa5\xd8\xb5\xd9\x84\xd8\xa7\xd8\xad\xd8\xa7\xd8\xaa \xd9\x86\xd9\x87\xd8\xa7\xd8\xa6\xd9\x8a\xd8\xa9:\n#   \xe2\x9c\x85 Zigzag Non-Repainting (Confirmed Swings \xd9\x81\xd9\x82\xd8\xb7)\n#   \xe2\x9c\x85 Pattern Confirmation Candle (neckline breakout)\n#   \xe2\x9c\x85 Elliott: W3 \xd8\xa3\xd8

#Train Patterns

In [ ]:


# ============================================================
# train_patterns.py  v3.4  —  Classical Pattern Recognition
# ============================================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
import joblib, os, json, logging, warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
np.random.seed(42)


# ── قراءة الداتا (Colab) ──────────────────────────────────────
def get_or_update_data(force_refresh=False):
    csv_path = "/content/XAUUSD_5min_raw.csv"
    df = pd.read_csv(csv_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    logging.info(f"CSV محمّل: {len(df):,} شمعة")
    return df


try:
    import talib as _talib
    HAS_TALIB = True
except ImportError:
    HAS_TALIB = False

SYMBOL_SAFE = "XAUUSD"
MODELS_DIR  = "models/patterns"
FIB_TOL     = 0.06

TIMEFRAME_CONFIG = {
    "5min": {
        "interval":"5min","years":2,"buy_threshold":0.62,"sell_threshold":0.38,
        "fold_buy_threshold":0.52,"fold_sell_threshold":0.48,
        "resample_rule":"5min","minutes_per_candle":5,
        "active_for_live":True,"min_candles":500,
        "horizon":12,"pattern_window":30,"min_return":0.001,"htf_resample":"1h",
        "zigzag_pct":0.004,"pattern_max_age":20,"confirmation_bars":2,"higher_tf":"1h",
        "min_pattern_quality":0.40,
        # v3.4: إعدادات Risk Management و Confidence
        "confidence_threshold":0.70,
        "atr_sl_mult":1.5,
        "atr_tp_mult":3.0,
    },
    "15min": {
        "interval":"15min","years":3,"buy_threshold":0.63,"sell_threshold":0.37,
        "fold_buy_threshold":0.52,"fold_sell_threshold":0.48,
        "resample_rule":"15min","minutes_per_candle":15,
        "active_for_live":True,"min_candles":500,
        "horizon":8,"pattern_window":20,"min_return":0.002,"htf_resample":"4h",
        "zigzag_pct":0.004,"pattern_max_age":20,"confirmation_bars":2,"higher_tf":"1h",
        "min_pattern_quality":0.38,
        "confidence_threshold":0.70,
        "atr_sl_mult":1.5,
        "atr_tp_mult":3.0,
    },
    "45min": {
        "interval":"45min","years":3,"buy_threshold":0.63,"sell_threshold":0.37,
        "fold_buy_threshold":0.52,"fold_sell_threshold":0.48,
        "resample_rule":"45min","minutes_per_candle":45,
        "active_for_live":True,"min_candles":300,
        "horizon":6,"pattern_window":15,"min_return":0.003,"htf_resample":"4h",
        "zigzag_pct":0.006,"pattern_max_age":15,"confirmation_bars":2,"higher_tf":"4h",
        "min_pattern_quality":0.35,
        "confidence_threshold":0.70,
        "atr_sl_mult":1.5,
        "atr_tp_mult":3.0,
    },
    "1h": {
        "interval":"1h","years":4,"buy_threshold":0.65,"sell_threshold":0.35,
        "fold_buy_threshold":0.52,"fold_sell_threshold":0.48,
        "resample_rule":"1h","minutes_per_candle":60,
        "active_for_live":True,"min_candles":500,
        "horizon":5,"pattern_window":15,"min_return":0.003,"htf_resample":"4h",
        "zigzag_pct":0.006,"pattern_max_age":15,"confirmation_bars":3,"higher_tf":"4h",
        "min_pattern_quality":0.30,
        "confidence_threshold":0.70,
        "atr_sl_mult":1.5,
        "atr_tp_mult":3.0,
    },
    "2h": {
        "interval":"2h","years":4,"buy_threshold":0.65,"sell_threshold":0.35,
        "fold_buy_threshold":0.52,"fold_sell_threshold":0.48,
        "resample_rule":"2h","minutes_per_candle":120,
        "active_for_live":True,"min_candles":300,
        "horizon":4,"pattern_window":12,"min_return":0.004,"htf_resample":"1d",
        "zigzag_pct":0.006,"pattern_max_age":12,"confirmation_bars":3,"higher_tf":"1d",
        "min_pattern_quality":0.30,
        "confidence_threshold":0.70,
        "atr_sl_mult":1.5,
        "atr_tp_mult":3.0,
    },
    "4h": {
        "interval":"4h","years":5,"buy_threshold":0.67,"sell_threshold":0.33,
        "fold_buy_threshold":0.51,"fold_sell_threshold":0.49,
        "resample_rule":"4h","minutes_per_candle":240,
        "active_for_live":True,"min_candles":200,
        "horizon":3,"pattern_window":12,"min_return":0.005,"htf_resample":"1d",
        "zigzag_pct":0.006,"pattern_max_age":12,"confirmation_bars":3,"higher_tf":"1d",
        "min_pattern_quality":0.28,
        "confidence_threshold":0.70,
        "atr_sl_mult":1.5,
        "atr_tp_mult":3.0,
    },
    "1d": {
        "interval":"1d","years":10,"buy_threshold":0.60,"sell_threshold":0.40,
        "fold_buy_threshold":0.52,"fold_sell_threshold":0.48,
        "resample_rule":"1B","minutes_per_candle":1440,
        "active_for_live":True,"min_candles":100,
        "horizon":3,"pattern_window":10,"min_return":0.008,"htf_resample":"W-FRI",
        "zigzag_pct":0.008,"pattern_max_age":10,"confirmation_bars":3,"higher_tf":"1W",
        "min_pattern_quality":0.25,
        "confidence_threshold":0.70,
        "atr_sl_mult":1.5,
        "atr_tp_mult":3.0,
    },
    "1W": {
        "interval":"1week","years":15,"buy_threshold":0.58,"sell_threshold":0.42,
        "fold_buy_threshold":0.51,"fold_sell_threshold":0.49,
        "resample_rule":"W-FRI","minutes_per_candle":10080,
        "active_for_live":True,"min_candles":80,
        "horizon":2,"pattern_window":8,"min_return":0.012,"htf_resample":None,
        "zigzag_pct":0.01,"pattern_max_age":8,"confirmation_bars":2,"higher_tf":None,
        "min_pattern_quality":0.22,
        "confidence_threshold":0.70,
        "atr_sl_mult":1.5,
        "atr_tp_mult":3.0,
    },
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler("training_patterns.log"), logging.StreamHandler()]
)


# ══════════════════════════════════════════════════════════════
#  ZIGZAG NON-REPAINTING
# ══════════════════════════════════════════════════════════════
def zigzag_swings_non_repaint(high, low, pct_thresh=0.01, confirmation_bars=3):
    highs = high.values if hasattr(high,'values') else np.asarray(high)
    lows  = low.values  if hasattr(low, 'values') else np.asarray(low)
    n = len(highs)
    if n < confirmation_bars + 3:
        return np.array([], dtype=int), np.array([], dtype=int)
    peak_idx, trough_idx = [], []
    direction = 1; last_high = 0; last_low = 0
    confirmed_idx_h = -1; confirmed_idx_l = -1
    for i in range(1, n):
        if direction == 1:
            if highs[i] > highs[last_high]: last_high = i
            if (lows[i] < highs[last_high]*(1.0-pct_thresh) and i-last_high >= confirmation_bars):
                if last_high > confirmed_idx_h:
                    peak_idx.append(last_high); confirmed_idx_h = last_high
                direction = -1; last_low = i
        else:
            if lows[i] < lows[last_low]: last_low = i
            if (highs[i] > lows[last_low]*(1.0+pct_thresh) and i-last_low >= confirmation_bars):
                if last_low > confirmed_idx_l:
                    trough_idx.append(last_low); confirmed_idx_l = last_low
                direction = 1; last_high = i
    return np.array(peak_idx, dtype=int), np.array(trough_idx, dtype=int)


def leg_ratio(a, b):
    return abs(b)/abs(a) if abs(a) > 1e-9 else 0.0


# ══════════════════════════════════════════════════════════════
# PATTERN FEATURE ENGINEER
# ══════════════════════════════════════════════════════════════
class PatternFeatureEngineer:
    def __init__(self):
        self.scaler          = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted   = False

    # ── Base Features ─────────────────────────────────────────
    def add_base_features(self, df):
        df = df.copy()
        c, h, l, o, v = df['close'], df['high'], df['low'], df['open'], df['volume']
        df['ema_20']  = c.ewm(span=20).mean()
        df['ema_50']  = c.ewm(span=50).mean()
        df['ema_200'] = c.ewm(span=200).mean()
        df['trend']   = np.sign(df['ema_20'] - df['ema_50'])
        tr = pd.concat([h-l,(h-c.shift()).abs(),(l-c.shift()).abs()],axis=1).max(axis=1)
        df['atr']     = tr.rolling(14).mean()
        df['atr_pct'] = df['atr']/(c+1e-9)
        delta = c.diff()
        ag = delta.clip(lower=0).rolling(14).mean()
        al = (-delta.clip(upper=0)).rolling(14).mean().clip(lower=1e-9)
        df['rsi']     = 100-(100/(1+ag/al))
        ema12 = c.ewm(span=12).mean(); ema26 = c.ewm(span=26).mean()
        df['macd']      = ema12-ema26
        df['macd_sig']  = df['macd'].ewm(span=9).mean()
        df['macd_hist'] = df['macd']-df['macd_sig']
        # ✅ vol_ratio بـ shifted vol_ma — منع data leakage
        vol_ma_s        = v.rolling(20).mean().shift(1).fillna(v.rolling(20).mean())
        df['vol_ratio'] = v/(vol_ma_s+1e-9)
        df['vol_surge'] = (df['vol_ratio']>2.0).astype(float)
        tp = (h+l+c)/3
        df['vwap']      = (tp*v).rolling(288,min_periods=1).sum()/(v.rolling(288,min_periods=1).sum()+1e-9)
        df['dist_vwap'] = (c-df['vwap'])/(df['atr']+1e-9)
        obv = (np.sign(c.diff())*v).fillna(0).cumsum()
        df['obv_slope'] = obv.diff(5)/(obv.shift(5).abs()+1e-9)
        df['returns']    = c.pct_change()
        df['volatility'] = df['returns'].rolling(20).std()
        df['body_pct']   = abs(c-o)/(h-l+1e-9)
        df['upper_wick'] = (h-c.clip(upper=o))/(h-l+1e-9)
        df['lower_wick'] = (c.clip(lower=o)-l)/(h-l+1e-9)
        df['hour']       = df['datetime'].dt.hour
        df['dow']        = df['datetime'].dt.dayofweek
        df['hour_sin']   = np.sin(2*np.pi*df['hour']/24)
        df['hour_cos']   = np.cos(2*np.pi*df['hour']/24)
        #  S/R distance بـ shifted high/low — منع data leakage
        res_s = h.shift(1).rolling(50,min_periods=1).max()
        sup_s = l.shift(1).rolling(50,min_periods=1).min()
        df['dist_to_resistance'] = (res_s-c)/(df['atr']+1e-9)
        df['dist_to_support']    = (c-sup_s)/(df['atr']+1e-9)
        df['near_breakout']      = ((df['dist_to_resistance']<1.0)|(df['dist_to_support']<1.0)).astype(float)
        df['sr_position']        = (c-sup_s)/(res_s-sup_s+1e-9)
        return df

    # ── HTF Trend ─────────────────────────────────────────────
    def add_htf_trend(self, df, htf_resample=None):
        df = df.copy()
        htf_cols = ['htf_trend','htf_ema_slope','htf_rsi','htf_above_ema200']
        for col in htf_cols: df[col] = 0.0
        if htf_resample is None: return df
        try:
            df_s = df.sort_values('datetime').reset_index(drop=True)
            df_s['datetime'] = pd.to_datetime(df_s['datetime'])
            htf = (df_s.set_index('datetime')
                   .resample(htf_resample)
                   .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'})
                   .dropna())
            if len(htf) < 10: return df
            htf = htf.reset_index()
            if 'datetime' not in htf.columns:
                htf = htf.rename(columns={htf.columns[0]:'datetime'})
            htf['datetime'] = pd.to_datetime(htf['datetime'])
            c = htf['close']
            htf['e50']  = c.ewm(span=50).mean()
            htf['e200'] = c.ewm(span=200).mean()
            htf['htf_trend']        = np.sign(htf['e50']-htf['e200'])
            htf['htf_ema_slope']    = htf['e50'].diff(3)/(htf['e50'].shift(3)+1e-9)
            htf['htf_above_ema200'] = (c>htf['e200']).astype(float)
            d=c.diff(); g=d.clip(lower=0).rolling(14).mean()
            ls=(-d.clip(upper=0)).rolling(14).mean().clip(lower=1e-9)
            htf['htf_rsi'] = 100-(100/(1+g/ls))
            slim = htf[['datetime','htf_trend','htf_ema_slope','htf_rsi','htf_above_ema200']].copy()
            slim = slim.sort_values('datetime').reset_index(drop=True)
            merged = pd.merge_asof(df_s, slim, on='datetime', direction='backward')
            for col in htf_cols:
                if col in merged.columns:
                    df_s[col] = merged[col].fillna(0).values
            for col in htf_cols:
                df[col] = df_s[col].fillna(0).values
        except Exception as e:
            logging.warning(f"HTF error: {e}")
        return df

    # ── Regime ────────────────────────────────────────────────
    def add_regime_features(self, df):
        df = df.copy()
        if HAS_TALIB:
            try:
                df['adx'] = _talib.ADX(df['high'].values, df['low'].values,
                                        df['close'].values, timeperiod=14)
            except Exception:
                df['adx'] = (df['atr']/(df['close']+1e-9)*500).clip(0,100)
        else:
            df['adx'] = (df['atr']/(df['close']+1e-9)*500).clip(0,100)
        atr_rank = df['atr'].rolling(100,min_periods=10).rank(pct=True)
        df['volatility_regime'] = np.select([atr_rank<0.33,atr_rank>0.66],[0,2],default=1)
        df['trend_strength']    = df['adx']/50.0
        df['market_regime']     = (df['adx']>25).astype(float)
        return df

    # ── Order Flow ────────────────────────────────────────────
    def add_order_flow(self, df):
        df = df.copy()
        mfv = ((df['close']-df['low'])-(df['high']-df['close']))/(df['high']-df['low']+1e-9)*df['volume']
        df['cmf'] = mfv.rolling(21).sum()/(df['volume'].rolling(21).sum()+1e-9)
        return df

    # ══════════════════════════════════════════════════════════
    #  CLASSIC PATTERNS
    # ══════════════════════════════════════════════════════════
    def detect_classic_patterns(self, df, window=20, zigzag_pct=0.01,
                                  max_age=20, confirmation_bars=3):
        df = df.copy()
        n  = len(df)
        hi = df['high'].values; lo = df['low'].values; cl = df['close'].values; vl = df['volume'].values

        cols = ['hs_pattern','hs_direction','hs_strength','hs_valid',
                'double_pattern','double_direction','double_strength','double_valid',
                'triple_pattern','triple_direction','triple_strength','triple_valid',
                'triangle_pattern','triangle_direction','triangle_strength',
                'flag_pattern','flag_direction','flag_strength',
                'wedge_pattern','wedge_direction','wedge_strength',
                'channel_pattern','channel_direction','channel_strength',
                'cup_handle_pattern','cup_handle_strength','cup_handle_valid',
                'rectangle_pattern','rectangle_direction','rectangle_strength',
                'pattern_age','pattern_volume_confirm']
        for col in cols: df[col] = 0.0

        #  vol_ma shifted لمنع data leakage
        vol_s  = pd.Series(vl)
        vol_ma = vol_s.rolling(20,min_periods=1).mean().shift(1).fillna(vol_s.rolling(20,min_periods=1).mean()).values
        atr_v  = df['atr'].values

        hi_idx, lo_idx = zigzag_swings_non_repaint(df['high'],df['low'],pct_thresh=zigzag_pct,confirmation_bars=confirmation_bars)
        swings = sorted(
            [{'idx':i,'price':hi[i],'type':'H'} for i in hi_idx]+
            [{'idx':i,'price':lo[i],'type':'L'} for i in lo_idx],
            key=lambda x: x['idx'])

        for si in range(2,len(swings)):
            pts   = swings[:si+1]; i = pts[-1]['idx']
            if i>=n or i+1>=n: continue
            a     = max(atr_v[i],1e-9)
            types = [p['type'] for p in pts]; prices=[p['price'] for p in pts]
            vc    = float(vl[i]/(vol_ma[i]+1e-9)>1.2)

            # H&S
            if len(pts)>=5:
                t5=types[-5:]; p5=prices[-5:]
                if t5==['H','L','H','L','H']:
                    sh_l,neck_l,head,neck_r,sh_r=p5; neck=(neck_l+neck_r)/2
                    if head>sh_l*1.005 and head>sh_r*1.005:
                        s=min(1.0,(head-max(sh_l,sh_r))/(a*3))*(1.3 if vc else 0.8)
                        df.at[i,'hs_pattern']=1.0; df.at[i,'hs_direction']=-1.0
                        df.at[i,'hs_strength']=min(1.0,s); df.at[i,'hs_valid']=float(cl[i]<neck)
                elif t5==['L','H','L','H','L']:
                    sh_l,neck_l,head,neck_r,sh_r=p5; neck=(neck_l+neck_r)/2
                    if head<sh_l*0.995 and head<sh_r*0.995:
                        s=min(1.0,(min(sh_l,sh_r)-head)/(a*3))*(1.3 if vc else 0.8)
                        df.at[i,'hs_pattern']=1.0; df.at[i,'hs_direction']=1.0
                        df.at[i,'hs_strength']=min(1.0,s); df.at[i,'hs_valid']=float(cl[i]>neck)

            # Double
            if len(pts)>=3:
                t3=types[-3:]; p3=prices[-3:]
                if t3==['H','L','H']:
                    h1,valley,h2=p3
                    if abs(h1-h2)/a<1.5:
                        s=(1.0-abs(h1-h2)/(a*1.5+1e-9))*(1.2 if vc else 0.8)
                        df.at[i,'double_pattern']=1.0; df.at[i,'double_direction']=-1.0
                        df.at[i,'double_strength']=min(1.0,s); df.at[i,'double_valid']=float(cl[i]<valley)
                elif t3==['L','H','L']:
                    l1,peak,l2=p3
                    if abs(l1-l2)/a<1.5:
                        s=(1.0-abs(l1-l2)/(a*1.5+1e-9))*(1.2 if vc else 0.8)
                        df.at[i,'double_pattern']=1.0; df.at[i,'double_direction']=1.0
                        df.at[i,'double_strength']=min(1.0,s); df.at[i,'double_valid']=float(cl[i]>peak)

            # Triple
            if len(pts)>=5:
                t5=types[-5:]; p5=prices[-5:]
                if t5==['H','L','H','L','H']:
                    h1,v1,h2,v2,h3=p5
                    if abs(h1-h2)/a<1.5 and abs(h2-h3)/a<1.5:
                        neck=min(v1,v2); s=max(0.0,1.0-(abs(h1-h2)+abs(h2-h3))/(a*3))*(1.2 if vc else 0.8)
                        df.at[i,'triple_pattern']=1.0; df.at[i,'triple_direction']=-1.0
                        df.at[i,'triple_strength']=min(1.0,s); df.at[i,'triple_valid']=float(cl[i]<neck)
                elif t5==['L','H','L','H','L']:
                    l1,p1,l2,p2,l3=p5
                    if abs(l1-l2)/a<1.5 and abs(l2-l3)/a<1.5:
                        neck=max(p1,p2); s=max(0.0,1.0-(abs(l1-l2)+abs(l2-l3))/(a*3))*(1.2 if vc else 0.8)
                        df.at[i,'triple_pattern']=1.0; df.at[i,'triple_direction']=1.0
                        df.at[i,'triple_strength']=min(1.0,s); df.at[i,'triple_valid']=float(cl[i]>neck)

            # Triangle / Wedge / Channel
            lh2=[p for p in pts if p['type']=='H'][-2:]
            ll2=[p for p in pts if p['type']=='L'][-2:]
            if len(lh2)==2 and len(ll2)==2:
                h_dx=max(1,lh2[-1]['idx']-lh2[-2]['idx']); l_dx=max(1,ll2[-1]['idx']-ll2[-2]['idx'])
                h_sl=(lh2[-1]['price']-lh2[-2]['price'])/h_dx; l_sl=(ll2[-1]['price']-ll2[-2]['price'])/l_dx
                if   h_sl<-1e-6 and l_sl>1e-6:
                    df.at[i,'triangle_pattern']=1.0; df.at[i,'triangle_direction']=0.0
                    df.at[i,'triangle_strength']=0.6*(1.1 if vc else 0.9)
                elif abs(h_sl)<1e-6 and l_sl>1e-6:
                    df.at[i,'triangle_pattern']=1.0; df.at[i,'triangle_direction']=1.0
                    df.at[i,'triangle_strength']=0.75*(1.2 if vc else 0.8)
                elif h_sl<-1e-6 and abs(l_sl)<1e-6:
                    df.at[i,'triangle_pattern']=1.0; df.at[i,'triangle_direction']=-1.0
                    df.at[i,'triangle_strength']=0.75*(1.2 if vc else 0.8)
                elif h_sl>1e-6 and l_sl>1e-6 and l_sl>h_sl:
                    df.at[i,'wedge_pattern']=1.0; df.at[i,'wedge_direction']=-1.0
                    df.at[i,'wedge_strength']=min(1.0,abs(l_sl-h_sl)/a*100*(1.2 if vc else 0.8))
                elif h_sl<-1e-6 and l_sl<-1e-6 and l_sl>h_sl:
                    df.at[i,'wedge_pattern']=1.0; df.at[i,'wedge_direction']=1.0
                    df.at[i,'wedge_strength']=min(1.0,abs(h_sl-l_sl)/a*100*(1.2 if vc else 0.8))
                sd=abs(h_sl-l_sl)/a
                if sd<0.3 and df.at[i,'triangle_pattern']==0 and df.at[i,'wedge_pattern']==0:
                    df.at[i,'channel_pattern']=1.0
                    df.at[i,'channel_direction']=1.0 if h_sl>0 else(-1.0 if h_sl<0 else 0.0)
                    df.at[i,'channel_strength']=max(0.0,1.0-sd)*(1.1 if vc else 0.9)

            # Flag
            if i>=window:
                pm=abs(cl[i]-cl[i-window])/a; rr=(max(hi[max(0,i-5):i+1])-min(lo[max(0,i-5):i+1]))/a
                if pm>5 and rr<2:
                    df.at[i,'flag_pattern']=1.0; df.at[i,'flag_direction']=1.0 if cl[i]>cl[i-window] else -1.0
                    df.at[i,'flag_strength']=min(1.0,pm/10)*(1.2 if vc else 0.8)

            # Cup & Handle
            if len(pts)>=3:
                t3=types[-3:]; p3=prices[-3:]
                if t3==['H','L','H']:
                    lh,cup_l,rh=p3; depth=(min(lh,rh)-cup_l)/a; rdiff=abs(lh-rh)/a
                    if depth>3 and rdiff<2 and cl[i]>cup_l*1.01:
                        rim=max(lh,rh)
                        df.at[i,'cup_handle_pattern']=1.0
                        df.at[i,'cup_handle_strength']=min(1.0,depth/10)*(1.3 if vc else 0.8)
                        df.at[i,'cup_handle_valid']=float(cl[i]>rim)

            # Rectangle
            l3h=[p['price'] for p in pts if p['type']=='H'][-3:]
            l3l=[p['price'] for p in pts if p['type']=='L'][-3:]
            if len(l3h)==3 and len(l3l)==3:
                hs2=np.std(l3h); ls2=np.std(l3l); ha=np.mean(l3h); la=np.mean(l3l)
                if hs2/a<1.0 and ls2/a<1.0 and la<cl[i]<ha:
                    df.at[i,'rectangle_pattern']=1.0; df.at[i,'rectangle_direction']=0.0
                    df.at[i,'rectangle_strength']=max(0.0,1.0-(hs2+ls2)/(a*2+1e-9))

            df.at[i,'pattern_volume_confirm']=vc

        # Pattern Age
        has_any=(df['hs_pattern']+df['double_pattern']+df['triple_pattern']+
                 df['triangle_pattern']+df['wedge_pattern']+df['flag_pattern']+
                 df['channel_pattern']+df['cup_handle_pattern']+df['rectangle_pattern'])>0
        count=999; bars=[]
        for v in has_any:
            count=0 if v else min(count+1,999); bars.append(count)
        df['pattern_age']=bars
        return df

    # ══════════════════════════════════════════════════════════
    #  HARMONIC PATTERNS — PRZ
    # ══════════════════════════════════════════════════════════
    def detect_harmonic_patterns(self, df, zigzag_pct=0.01, confirmation_bars=3):
        df=df.copy(); n=len(df)
        for col in ['harmonic_pattern','harmonic_type','harmonic_direction',
                    'harmonic_strength','harmonic_prz','harmonic_prz_high','harmonic_prz_low','harmonic_valid']:
            df[col]=0.0
        PATS={
            'gartley':  {'AB_XA':(0.618,0.618),'BC_AB':(0.382,0.886),'CD_BC':(1.272,1.618),'AD_XA':(0.786,0.786),'sc':4,'prz_fib':0.786},
            'butterfly':{'AB_XA':(0.786,0.786),'BC_AB':(0.382,0.886),'CD_BC':(1.618,2.618),'AD_XA':(1.272,1.618),'sc':3,'prz_fib':1.272},
            'bat':      {'AB_XA':(0.382,0.500),'BC_AB':(0.382,0.886),'CD_BC':(1.618,2.618),'AD_XA':(0.886,0.886),'sc':3,'prz_fib':0.886},
            'crab':     {'AB_XA':(0.382,0.618),'BC_AB':(0.382,0.886),'CD_BC':(2.240,3.618),'AD_XA':(1.618,1.618),'sc':2,'prz_fib':1.618},
            'shark':    {'AB_XA':(0.446,0.618),'BC_AB':(1.130,1.618),'CD_BC':(0.886,1.130),'AD_XA':(0.886,1.130),'sc':2,'prz_fib':0.886},
            'cypher':   {'AB_XA':(0.382,0.618),'BC_AB':(1.130,1.414),'CD_BC':(0.786,0.786),'AD_XA':(0.786,0.786),'sc':2,'prz_fib':0.786},
        }
        hi_idx,lo_idx=zigzag_swings_non_repaint(df['high'],df['low'],pct_thresh=zigzag_pct,confirmation_bars=confirmation_bars)
        swings=sorted([{'idx':i,'price':df['high'].iloc[i],'type':'H'} for i in hi_idx]+
                      [{'idx':i,'price':df['low'].iloc[i],'type':'L'} for i in lo_idx],key=lambda x:x['idx'])
        closes=df['close'].values
        for i in range(4,len(swings)):
            pts=swings[i-4:i+1]; types=[p['type'] for p in pts]
            prices=[p['price'] for p in pts]; idxs=[p['idx'] for p in pts]; Di=idxs[4]
            if Di>=n: continue
            is_bear=(types==['H','L','H','L','H']); is_bull=(types==['L','H','L','H','L'])
            if not(is_bear or is_bull): continue
            X,A,B,C,D=prices
            XA=abs(A-X);AB=abs(B-A);BC=abs(C-B);CD=abs(D-C);AD=abs(D-A)
            if any(v<1e-9 for v in [XA,AB,BC,CD]): continue
            ab_xa=leg_ratio(XA,AB);bc_ab=leg_ratio(AB,BC);cd_bc=leg_ratio(BC,CD);ad_xa=leg_ratio(XA,AD)
            t=FIB_TOL
            for pn,r in PATS.items():
                la,ha=r['AB_XA'];lb,hb=r['BC_AB'];lc,hc=r['CD_BC'];ld,hd=r['AD_XA']
                if la-t<=ab_xa<=ha+t and lb-t<=bc_ab<=hb+t and lc-t<=cd_bc<=hc+t and ld-t<=ad_xa<=hd+t:
                    acc=1.0-(abs(ab_xa-(la+ha)/2)/(ha-la+0.01)+abs(bc_ab-(lb+hb)/2)/(hb-lb+0.01)+abs(cd_bc-(lc+hc)/2)/(hc-lc+0.01))/3
                    prz_c=(X+(A-X)*r['prz_fib']) if is_bull else (X-(X-A)*r['prz_fib'])
                    atr_d=max(df['atr'].iloc[Di],1e-9)
                    df.at[Di,'harmonic_pattern']=1.0; df.at[Di,'harmonic_type']=float(r['sc'])
                    df.at[Di,'harmonic_direction']=-1.0 if is_bear else 1.0
                    df.at[Di,'harmonic_strength']=max(0.0,min(1.0,acc))
                    df.at[Di,'harmonic_prz']=1.0
                    df.at[Di,'harmonic_prz_high']=prz_c+atr_d*0.5
                    df.at[Di,'harmonic_prz_low']=prz_c-atr_d*0.5
                    df.at[Di,'harmonic_valid']=float((prz_c-atr_d*0.5)<=closes[Di]<=(prz_c+atr_d*0.5))
                    break
        return df

    # ══════════════════════════════════════════════════════════
    #  ELLIOTT WAVE —
    # ══════════════════════════════════════════════════════════
    def detect_elliott_wave(self, df, zigzag_pct=0.01, confirmation_bars=3):
        df=df.copy(); n=len(df)
        ew_cols=['elliott_impulse','elliott_wave_num','elliott_corrective','elliott_wave3',
                 'elliott_bullish','elliott_strength','elliott_extended_w3','elliott_diagonal',
                 'elliott_valid','elliott_w1_len','elliott_w3_len','elliott_w5_len']
        for col in ew_cols: df[col]=0.0
        hi_idx,lo_idx=zigzag_swings_non_repaint(df['high'],df['low'],pct_thresh=zigzag_pct,confirmation_bars=confirmation_bars)
        swings=sorted([{'idx':i,'price':df['high'].iloc[i],'type':'H'} for i in hi_idx]+
                      [{'idx':i,'price':df['low'].iloc[i],'type':'L'} for i in lo_idx],key=lambda x:x['idx'])
        atr=df['atr'].values

        for i in range(4,len(swings)):
            pts=swings[i-4:i+1]; types=[p['type'] for p in pts]
            prices=[p['price'] for p in pts]; idxs=[p['idx'] for p in pts]

            # ── Bullish Impulse ──
            if types==['L','H','L','H','L']:
                w0,w1,w2,w3,w4=prices
                l1=w1-w0; l2=w1-w2; l3=w3-w2; l4=w3-w4; l5=abs(prices[4]-w4)
                valid=True
                if l1<=0 or l3<=0:             valid=False
                if w2<=w0:                      valid=False  # W2 لا تتخطى W0
                if w4<=w1:                      valid=False  # W4 Overlap Rule
                if l3<l1*0.9:                   valid=False  # W3 ليست أقصر
                w2_r=l2/(l1+1e-9)
                if not(0.236<=w2_r<=0.886):     valid=False  # W2 Fibonacci
                if valid:
                    w4_r=l4/(l3+1e-9)
                    ext=l3>l1*1.618; alt=not(w2_r>0.5 and w4_r>0.5); w5ok=abs(l5-l1)/(l1+1e-9)<0.30
                    qb=1.0+(0.20 if ext else 0)+(0.15 if alt else 0)+(0.10 if w5ok else 0)
                    ci=idxs[-1]
                    if ci<n:
                        ai=max(atr[ci],1e-9)
                        df.at[ci,'elliott_impulse']=1.0; df.at[ci,'elliott_wave_num']=5.0
                        df.at[ci,'elliott_bullish']=1.0; df.at[ci,'elliott_valid']=1.0
                        df.at[ci,'elliott_strength']=min(1.0,l3/(ai*5)*qb)
                        df.at[ci,'elliott_w1_len']=l1/ai; df.at[ci,'elliott_w3_len']=l3/ai; df.at[ci,'elliott_w5_len']=l5/ai
                    w3i=idxs[3]
                    if w3i<n:
                        aw=max(atr[w3i],1e-9)
                        df.at[w3i,'elliott_wave3']=1.0; df.at[w3i,'elliott_impulse']=1.0
                        df.at[w3i,'elliott_wave_num']=3.0; df.at[w3i,'elliott_bullish']=1.0
                        df.at[w3i,'elliott_valid']=1.0
                        df.at[w3i,'elliott_strength']=min(1.0,l3/(aw*3)*qb)
                        df.at[w3i,'elliott_extended_w3']=float(ext)

            # ── Bearish Impulse ──
            elif types==['H','L','H','L','H']:
                w0,w1,w2,w3,w4=prices
                l1=w0-w1; l2=w2-w1; l3=w2-w3; l4=w4-w3; l5=abs(prices[4]-w4)
                valid=True
                if l1<=0 or l3<=0:             valid=False
                if w2>=w0:                      valid=False
                if w4>=w1:                      valid=False
                if l3<l1*0.9:                   valid=False
                w2_r=l2/(l1+1e-9)
                if not(0.236<=w2_r<=0.886):     valid=False
                if valid:
                    w4_r=l4/(l3+1e-9)
                    ext=l3>l1*1.618; alt=not(w2_r>0.5 and w4_r>0.5); w5ok=abs(l5-l1)/(l1+1e-9)<0.30
                    qb=1.0+(0.20 if ext else 0)+(0.15 if alt else 0)+(0.10 if w5ok else 0)
                    ci=idxs[-1]
                    if ci<n:
                        ai=max(atr[ci],1e-9)
                        df.at[ci,'elliott_impulse']=1.0; df.at[ci,'elliott_wave_num']=5.0
                        df.at[ci,'elliott_bullish']=0.0; df.at[ci,'elliott_valid']=1.0
                        df.at[ci,'elliott_strength']=min(1.0,l3/(ai*5)*qb)
                        df.at[ci,'elliott_w1_len']=l1/ai; df.at[ci,'elliott_w3_len']=l3/ai; df.at[ci,'elliott_w5_len']=l5/ai
                    w3i=idxs[3]
                    if w3i<n:
                        aw=max(atr[w3i],1e-9)
                        df.at[w3i,'elliott_wave3']=1.0; df.at[w3i,'elliott_impulse']=1.0
                        df.at[w3i,'elliott_wave_num']=3.0; df.at[w3i,'elliott_bullish']=0.0
                        df.at[w3i,'elliott_valid']=1.0
                        df.at[w3i,'elliott_strength']=min(1.0,l3/(aw*3)*qb)
                        df.at[w3i,'elliott_extended_w3']=float(ext)

            # ── Diagonal ──
            if types in [['L','H','L','H','L'],['H','L','H','L','H']]:
                legs=[abs(prices[j+1]-prices[j]) for j in range(4)]
                if all(legs[j]<legs[j-1] for j in range(1,4)):
                    ci=idxs[-1]
                    if ci<n:
                        df.at[ci,'elliott_diagonal']=1.0
                        df.at[ci,'elliott_bullish']=float(types==['L','H','L','H','L'])
                        df.at[ci,'elliott_valid']=1.0

        # ── Corrective A-B-C ──
        for i in range(2,len(swings)):
            pts=swings[i-2:i+1]; types=[p['type'] for p in pts]
            prices=[p['price'] for p in pts]; ci=pts[-1]['idx']
            if ci>=n: continue
            if types==['H','L','H']:
                A,B,C=prices; lA=A-B; lB=C-B
                if lA>0 and lB>0 and 0.382<=lB/(lA+1e-9)<=1.618:
                    df.at[ci,'elliott_corrective']=1.0; df.at[ci,'elliott_bullish']=1.0; df.at[ci,'elliott_valid']=1.0
            elif types==['L','H','L']:
                A,B,C=prices; lA=B-A; lB=B-C
                if lA>0 and lB>0 and 0.382<=lB/(lA+1e-9)<=1.618:
                    df.at[ci,'elliott_corrective']=1.0; df.at[ci,'elliott_bullish']=0.0; df.at[ci,'elliott_valid']=1.0
        return df

    # ══════════════════════════════════════════════════════════
    #  PATTERN META — safe helper لمنع NaN/KeyError
    # ══════════════════════════════════════════════════════════
    def add_pattern_meta_classical(self, df, max_age=20):
        df=df.copy()

        def _s(col, w=1.0, vcol=None):
            """Safe get: يسترجع عمود بأمان ويضرب في الـ valid لو موجود"""
            base = df[col].fillna(0) if col in df.columns else pd.Series(0.0,index=df.index)
            if vcol:
                vv = df[vcol].fillna(0) if vcol in df.columns else pd.Series(1.0,index=df.index)
                return base*w*vv
            return base*w

        pcols=['hs_pattern','double_pattern','triple_pattern','triangle_pattern','flag_pattern',
               'wedge_pattern','channel_pattern','cup_handle_pattern','rectangle_pattern',
               'harmonic_pattern','elliott_impulse','elliott_corrective']
        df['pattern_count'] = df[pcols].clip(0,1).sum(axis=1)

        #  elliott_extended_w3 safe
        ew3  = df['elliott_wave3'].clip(0,1) if 'elliott_wave3' in df.columns else pd.Series(0.0,index=df.index)
        ewx3 = df['elliott_extended_w3'].clip(0,1) if 'elliott_extended_w3' in df.columns else pd.Series(0.0,index=df.index)
        df['pattern_cluster'] = (_s('harmonic_pattern')+_s('hs_pattern')+_s('double_pattern')+_s('triple_pattern')+ew3+ewx3)
        df['strong_cluster']  = (df['pattern_cluster']>=2).astype(float)

        # Pattern Quality
        df['pattern_quality'] = (
            _s('harmonic_strength',3.0,'harmonic_valid')+_s('hs_strength',2.5,'hs_valid')+
            _s('elliott_strength',2.2,'elliott_valid')+_s('triple_strength',2.0,'triple_valid')+
            _s('cup_handle_strength',1.8,'cup_handle_valid')+_s('double_strength',1.5,'double_valid')+
            _s('triangle_strength',1.3)+_s('flag_strength',1.0)+_s('wedge_strength',1.0)+
            _s('channel_strength',0.8)+_s('rectangle_strength',0.6)
        )
        qmax=df['pattern_quality'].max()
        df['pattern_quality']=df['pattern_quality']/(qmax+1e-9)

        # Pattern Valid
        vcols=[c for c in ['hs_valid','double_valid','triple_valid','cup_handle_valid','harmonic_valid','elliott_valid'] if c in df.columns]
        df['pattern_valid']=(df[vcols].sum(axis=1)>0).astype(float) if vcols else pd.Series(0.0,index=df.index)

        # Direction Consensus
        dcols=[c for c in ['hs_direction','double_direction','triple_direction','triangle_direction','harmonic_direction','elliott_bullish'] if c in df.columns]
        df['direction_consensus']=df[dcols].mean(axis=1) if dcols else 0.0

        # Age & Freshness
        df['bars_since_pattern']=df['pattern_age']
        df['pattern_freshness']=1.0/(df['bars_since_pattern']+1)
        df['pattern_expired']=(df['pattern_age']>max_age).astype(float)

        # HTF Alignment
        df['pattern_htf_aligned']=(
            ((df['direction_consensus']>0)&(df['htf_trend']>0))|
            ((df['direction_consensus']<0)&(df['htf_trend']<0))
        ).astype(float)

        # Volume
        if 'pattern_volume_confirm' in df.columns:
            df['pattern_with_volume']=((df['pattern_count']>=1)&(df['pattern_volume_confirm']==1)).astype(float)
        else:
            df['pattern_with_volume']=0.0

        # Pattern Score
        df['pattern_score']=(
            df['pattern_quality']*df['pattern_freshness']*
            (1.0+df['pattern_htf_aligned']*0.6)*(1.0+df['pattern_with_volume']*0.3)*(1.0+df['pattern_valid']*0.5)
        )

        #  PRZ Shifted لتجنب lookahead
        if 'harmonic_prz_high' in df.columns:
            prz_h=df['harmonic_prz_high'].shift(1).fillna(0.0)
            prz_l=df['harmonic_prz_low'].shift(1).fillna(0.0)
            df['in_prz']=((df['close']>=prz_l)&(df['close']<=prz_h)&(prz_h>0)).astype(float)
            df['dist_to_prz']=(df['close']-(prz_h+prz_l)/2).abs()/(df['atr']+1e-9)
        else:
            df['in_prz']=0.0; df['dist_to_prz']=0.0
        return df

    # ── create_features ───────────────────────────────────────
    def create_features(self, df, config, shift=True):
        w=config.get('pattern_window',20); hr=config.get('htf_resample',None)
        zp=config.get('zigzag_pct',0.01); ma=config.get('pattern_max_age',20)
        cb=config.get('confirmation_bars',3)
        df=df.reset_index(drop=True).copy()
        df=self.add_base_features(df); df=self.add_htf_trend(df,htf_resample=hr)
        df=self.add_regime_features(df); df=self.add_order_flow(df)
        df=self.detect_classic_patterns(df,window=w,zigzag_pct=zp,max_age=ma,confirmation_bars=cb)
        df=self.detect_harmonic_patterns(df,zigzag_pct=zp,confirmation_bars=cb)
        df=self.detect_elliott_wave(df,zigzag_pct=zp,confirmation_bars=cb)
        df=self.add_pattern_meta_classical(df,max_age=ma)
        excl=['datetime','open','high','low','close','volume','hour','dow']
        if shift:
            for col in [c for c in df.columns if c not in excl]:
                df[col]=df[col].shift(1)
        thresh=int(len(df)*0.5); df=df.dropna(axis=1,thresh=thresh)
        warmup=max(1,len(df)//10); df=df.iloc[warmup:].copy(); df=df.ffill().fillna(0)
        excl_f=['datetime','open','high','low','close','volume','future_close','pct_change','target']
        self.feature_columns=[c for c in df.columns if c not in excl_f]
        logging.info(f"Features: {len(self.feature_columns)}")
        return df

    # ── create_target ─────────────────────────────────────────
    def create_target(self, df, horizon=5, min_return=0.003):
        df=df.copy()
        df['future_close']=df['close'].shift(-horizon)
        df['pct_change']=(df['future_close']-df['close'])/(df['close']+1e-9)
        pct=df['pct_change'].dropna()
        if 'atr' in df.columns and len(pct)>50:
            thr=max((df['atr']/df['close'].replace(0,1e-9)).median()*0.8, min_return)
        elif len(pct)>50: thr=pct.quantile(0.65)
        else: thr=min_return
        if len(pct)>50:
            cand=(pct>thr).mean()
            if cand<0.20: thr=pct.quantile(0.75)
            elif cand>0.50: thr=pct.quantile(0.55)
        df['target']=(df['pct_change']>thr).astype(int)
        df.dropna(subset=['future_close'],inplace=True)
        logging.info(f"Target: {len(df)} صف | Pos:{df['target'].mean()*100:.1f}% | thr={thr:.5f}")
        return df

    def fit_scaler(self,X):
        self.scaler.fit(X); self.scaler_fitted=True
        if isinstance(X,pd.DataFrame): self.feature_columns=list(X.columns)
    def transform(self,X):
        if not self.scaler_fitted: raise RuntimeError("scaler not fitted")
        if isinstance(X,pd.DataFrame): X=X[self.feature_columns]
        return self.scaler.transform(X)
    def fit_transform(self,X):
        self.fit_scaler(X); return self.transform(X)


# ══════════════════════════════════════════════════════════════
#  MULTI-TIMEFRAME DECISION LAYER
# ══════════════════════════════════════════════════════════════
def multi_timeframe_decision(signals: dict) -> dict:
    if not signals:
        return {"final_signal":"NO TRADE","confidence":0.0,"reason":"لا إشارات"}
    TF_W={"1W":5,"1d":4,"4h":3,"2h":2.5,"1h":2,"45min":1.5,"15min":1,"5min":0.5}
    dirs={tf:v.get('direction',0) for tf,v in signals.items()}
    probs={tf:v.get('prob',0.5) for tf,v in signals.items()}
    qualities={tf:v.get('pattern_quality',0.0) for tf,v in signals.items()}
    valids={tf:v.get('pattern_valid',0.0) for tf,v in signals.items()}
    htf_t={tf:v.get('htf_trend',0) for tf,v in signals.items()}
    wd=tw=0.0
    for tf,d in dirs.items():
        w=TF_W.get(tf,1.0); wd+=d*w*probs.get(tf,0.5); tw+=w
    if tw>0: wd/=tw
    avg_q=float(np.mean(list(qualities.values()))) if qualities else 0.0
    avg_p=float(np.mean(list(probs.values()))) if probs else 0.5
    avg_v=float(np.mean(list(valids.values()))) if valids else 0.0
    htf_ok=all(htf_t.get(tf,0)*dirs.get(tf,0)>=0 for tf in signals if dirs.get(tf,0)!=0)
    if avg_q<0.3:
        return {"final_signal":"NO TRADE","confidence":0.0,"reason":f"جودة منخفضة ({avg_q:.2f})"}
    if avg_p>0.80 and avg_q>0.6 and avg_v>0.5: cl="strong"
    elif avg_p>0.65 and avg_q>0.4: cl="medium"
    else: cl="weak"
    all_buy=all(d>0 for d in dirs.values() if d!=0)
    all_sell=all(d<0 for d in dirs.values() if d!=0)
    maj_buy=wd>0.3; maj_sell=wd<-0.3
    reason=["⚠️ HTF غير متوافق"] if not htf_ok else []
    if all_buy and htf_ok and cl=="strong":   sig="STRONG BUY 🔥";  conf=min(0.95,avg_p); reason.append("كل الفريمات BUY")
    elif all_sell and htf_ok and cl=="strong": sig="STRONG SELL 🔥"; conf=min(0.95,avg_p); reason.append("كل الفريمات SELL")
    elif maj_buy and htf_ok and cl!="weak":   sig="BUY 📊";          conf=avg_p*0.85
    elif maj_sell and htf_ok and cl!="weak":  sig="SELL 📊";         conf=avg_p*0.85
    elif maj_buy and not htf_ok:              sig="WEAK BUY ⚠️";    conf=avg_p*0.5
    elif maj_sell and not htf_ok:             sig="WEAK SELL ⚠️";   conf=avg_p*0.5
    else:                                      sig="NO TRADE ❌";     conf=0.0; reason.append(f"تعارض (wd={wd:.2f})")
    return {"final_signal":sig,"confidence":round(conf,3),"confidence_level":cl,
            "weighted_direction":round(wd,3),"avg_quality":round(avg_q,3),"avg_valid":round(avg_v,3),
            "avg_prob":round(avg_p,3),"htf_aligned":htf_ok,"reason":" | ".join(reason),"tf_signals":signals}


def get_signal_confidence_label(prob: float) -> str:
    if prob>0.80: return "STRONG 🔥"
    elif prob>0.65: return "MEDIUM 📊"
    elif prob>0.52: return "WEAK ⚠️"
    else: return "NO SIGNAL ❌"


# ══════════════════════════════════════════════════════════════
#  BACKTESTER v3.4  —  مع إدارة المخاطر واستخدام الثقة
# ══════════════════════════════════════════════════════════════
class Backtester:
    def __init__(self, initial_capital=10000, spread_bps=2,
                 risk_per_trade=0.005, max_positions=1,
                 max_position_pct=0.05, max_drawdown_pct=0.20,
                 min_pattern_quality=0.30, require_valid=False,
                 confidence_threshold=0.70, atr_sl_mult=1.5, atr_tp_mult=3.0):
        """
        confidence_threshold: قيمة الثقة المطلوبة لتنفيذ الصفقة (مثلاً 0.7)
        atr_sl_mult: مضاعف ATR لوضع Stop Loss
        atr_tp_mult: مضاعف ATR لوضع Take Profit
        """
        self.initial_capital=initial_capital; self.spread=spread_bps/10000
        self.slippage=0.0001; self.risk_per_trade=risk_per_trade
        self.max_positions=max_positions; self.max_position_pct=max_position_pct
        self.max_drawdown_pct=max_drawdown_pct; self.min_pattern_quality=min_pattern_quality
        self.require_valid=require_valid; self.max_dd_limit=initial_capital*0.30
        self.confidence_threshold = confidence_threshold
        self.atr_sl_mult = atr_sl_mult
        self.atr_tp_mult = atr_tp_mult

    def _exec(self,p,buy):
        return p*(1+self.spread)*(1+self.slippage) if buy else p*(1-self.spread)*(1-self.slippage)

    def run(self,df,model,fe,feature_columns,cfg,atr_sl=None,atr_tp=None,use_fold=False):
        """
        يستخدم الثقة المحددة في التهيئة (confidence_threshold) لتحديد buy/sell thresholds
        ما لم يتم تمرير buy_threshold/sell_threshold في cfg و use_fold=True (للمقارنة).
        """
        capital=self.initial_capital; positions=[]; trades=[]; equity=[capital]
        missing=[c for c in feature_columns+['close','atr'] if c not in df.columns]
        if missing: logging.error(f"Missing: {missing}"); return {}

        # تحديد thresholds بناءً على الثقة أو القيم المخزنة في cfg
        if use_fold and cfg.get('fold_buy_threshold') is not None:
            bth = cfg['fold_buy_threshold']
            sth = cfg['fold_sell_threshold']
        elif not use_fold and cfg.get('buy_threshold') is not None:
            bth = cfg['buy_threshold']
            sth = cfg['sell_threshold']
        else:
            # استخدام الثقة المحددة في الـ Backtester
            bth = self.confidence_threshold
            sth = 1.0 - self.confidence_threshold

        # استخدام مضاعفات ATR من الـ Backtester (أو من cfg)
        sl_mult = self.atr_sl_mult if atr_sl is None else atr_sl
        tp_mult = self.atr_tp_mult if atr_tp is None else atr_tp

        proba=model.predict_proba(fe.transform(df[feature_columns].copy()))[:,1]
        closes=df['close'].values; atrs=df['atr'].values
        pq=df['pattern_quality'].values if 'pattern_quality' in df.columns else np.full(len(df),1.0)
        pv=df['pattern_valid'].values   if 'pattern_valid'   in df.columns else np.ones(len(df))
        htf=df['htf_trend'].values       if 'htf_trend'       in df.columns else np.zeros(len(df))
        adx_v=df['adx'].values           if 'adx'             in df.columns else np.full(len(df),25.0)
        adx_min=10.0; peak_capital=capital

        for i in range(len(df)):
            p,a,pr=closes[i],atrs[i],proba[i]
            peak_capital=max(peak_capital,capital)
            curr_dd=(capital-peak_capital)/(peak_capital+1e-9)
            if curr_dd<-self.max_drawdown_pct:
                sig="HOLD"
            else:
                raw="BUY" if pr>bth else("SELL" if pr<sth else "HOLD")
                # تصفية إضافية باستخدام pattern_quality و pattern_valid و adx و htf
                if   raw!="HOLD" and pq[i]<self.min_pattern_quality:         sig="HOLD"
                elif raw!="HOLD" and self.require_valid and pv[i]<1.0:        sig="HOLD"
                elif raw!="HOLD" and adx_v[i]<adx_min:                        sig="HOLD"
                elif raw=="BUY"  and htf[i]<0:                                 sig="HOLD"
                elif raw=="SELL" and htf[i]>0:                                 sig="HOLD"
                else: sig=raw

            to_close=[]
            for pos in positions:
                if pos['type']=='buy':
                    if p<=pos['sl']:
                        capital+=pos['units']*pos['sl']
                        profit = pos['units']*(pos['sl']-pos['entry'])
                        trades.append({'profit':profit,'exit':'sl','direction':1,'entry':pos['entry'],'exit_price':pos['sl']})
                        to_close.append(pos)
                    elif p>=pos['tp']:
                        capital+=pos['units']*pos['tp']
                        profit = pos['units']*(pos['tp']-pos['entry'])
                        trades.append({'profit':profit,'exit':'tp','direction':1,'entry':pos['entry'],'exit_price':pos['tp']})
                        to_close.append(pos)
                else:
                    if p>=pos['sl']:
                        pnl = pos['units']*(pos['entry']-pos['sl'])
                        capital+=pos['margin']+pnl
                        profit = pnl  # negative if loss
                        trades.append({'profit':profit,'exit':'sl','direction':-1,'entry':pos['entry'],'exit_price':pos['sl']})
                        to_close.append(pos)
                    elif p<=pos['tp']:
                        pnl = pos['units']*(pos['entry']-pos['tp'])
                        capital+=pos['margin']+pnl
                        profit = pnl
                        trades.append({'profit':profit,'exit':'tp','direction':-1,'entry':pos['entry'],'exit_price':pos['tp']})
                        to_close.append(pos)
            for pos in to_close: positions.remove(pos)

            if sig=="BUY" and len(positions)<self.max_positions and a>0:
                ra=capital*self.risk_per_trade; sd=sl_mult*a; units=ra/sd if sd>0 else 0
                if units>0:
                    entry=self._exec(p,True); mc=capital*self.max_position_pct
                    cost=min(units*entry,mc); units=cost/entry
                    if cost>0 and cost<=capital:
                        capital-=cost
                        positions.append({'type':'buy','entry':entry,'units':units,'sl':entry-sd,'tp':entry+tp_mult*a,'margin':cost})
            elif sig=="SELL" and len(positions)<self.max_positions and a>0:
                ra=capital*self.risk_per_trade; sd=sl_mult*a; units=ra/sd if sd>0 else 0
                if units>0:
                    entry=self._exec(p,False); margin=min(ra,capital*self.max_position_pct)
                    if margin>0 and margin<=capital:
                        capital-=margin
                        positions.append({'type':'sell','entry':entry,'units':units,'sl':entry+sd,'tp':entry-tp_mult*a,'margin':margin})

            ov=sum(pos['units']*p-pos['units']*pos['entry'] if pos['type']=='buy'
                   else pos['units']*pos['entry']-pos['units']*p for pos in positions)
            equity.append(capital+ov)

        # إغلاق المراكز المتبقية عند نهاية الداتا
        for pos in positions:
            lp=closes[-1]
            if pos['type']=='buy':
                pnl = pos['units']*(self._exec(lp,False)-pos['entry'])
                capital+=pos['units']*self._exec(lp,False)
            else:
                pnl = pos['units']*(pos['entry']-self._exec(lp,True))
                capital+=pos['margin']+pnl
            trades.append({'profit':pnl,'exit':'eod','direction':1 if pos['type']=='buy' else -1,
                           'entry':pos['entry'],'exit_price':lp})

        if not trades: return {}
        wins=[t for t in trades if t['profit']>0]; loss=[t for t in trades if t['profit']<=0]
        wr=len(wins)/len(trades)*100; tp_s=sum(t['profit'] for t in wins)
        tl_s=abs(sum(t['profit'] for t in loss))
        aw=tp_s/len(wins) if wins else 0; al=tl_s/len(loss) if loss else 0
        pf=tp_s/tl_s if tl_s>0 else float('inf')
        ret=(capital-self.initial_capital)/self.initial_capital*100
        eq_s=pd.Series(equity); dd=float(((eq_s-eq_s.cummax())/(eq_s.cummax()+1e-9)*100).min())
        return {'total_trades':len(trades),'win_rate':round(wr,2),'profit_factor':round(pf,2),
                'total_return_pct':round(ret,2),'max_drawdown_pct':round(dd,2),
                'avg_win':round(aw,2),'avg_loss':round(al,2),
                'expectancy':round((wr/100*aw)-((1-wr/100)*al),2) if al else 0,
                'trades':trades}   # إضافة trades للاطلاع على تفاصيل الربح/الخسارة


# ── XGB ──────────────────────────────────────────────────────
def _build_xgb(spw, n_est=1000):
    return xgb.XGBClassifier(
        objective='binary:logistic',n_estimators=n_est,max_depth=6,
        learning_rate=0.02,subsample=0.8,colsample_bytree=0.65,
        reg_alpha=0.5,reg_lambda=2.0,min_child_weight=5,gamma=0.1,
        scale_pos_weight=spw,random_state=42,tree_method='hist',
        verbosity=0,use_label_encoder=False,eval_metric='logloss',
        early_stopping_rounds=50)


# ══════════════════════════════════════════════════════════════
#  WALK-FORWARD — tf_name parameter (منع NameError)
# ══════════════════════════════════════════════════════════════
def walk_forward_expanding(X, y, df_tgt, cfg, n_splits=6, purge_pct=0.01, tf_name=''):
    n=len(X); purge_size=max(1,int(n*purge_pct))
    min_init=min(200,max(n//3,10)); init_train=max(int(n*0.40),min_init)
    remaining=n-init_train; test_size=max(int(remaining/n_splits),max(10,n//20))
    #  use_valid وmin_pq من cfg — مش من متغير خارجي
    use_valid=tf_name in('4h','1d','1W','2h')
    min_pq=cfg.get('min_pattern_quality',0.30)
    logging.info(f"  WF: n={n}|init={init_train}|test={test_size}|purge={purge_size}|valid={use_valid}|min_pq={min_pq}")
    fold_results=[]; best_model,best_score,best_fe=None,-np.inf,None; mdl=None; fef=None

    for fold in range(n_splits):
        train_end=init_train+fold*test_size; test_start=train_end+purge_size; test_end=min(test_start+test_size,n)
        if test_start>=n or test_end<=test_start: break
        tei=list(range(test_start,test_end))
        if len(y.iloc[tei].unique())<2:
            ext_end=min(test_end+test_size,n); tei_ext=list(range(test_start,ext_end))
            if len(y.iloc[tei_ext].unique())>=2: tei=tei_ext; logging.warning(f"  Fold {fold+1}: توسيع → {len(tei)}")
            else: logging.warning(f"  Fold {fold+1}: فئة واحدة — تخطي"); continue
        tri=list(range(0,train_end))
        Xtr,Xte=X.iloc[tri],X.iloc[tei]; ytr,yte=y.iloc[tri],y.iloc[tei]
        if len(ytr.unique())<2:
            mc=1 if(ytr==0).all() else 0
            Xtr=pd.concat([Xtr,Xte.iloc[:1]]); ytr=pd.concat([ytr,pd.Series([mc],index=[Xte.index[0]])])
        neg=(ytr==0).sum(); pos=(ytr==1).sum(); spw=neg/pos if pos>0 else 1.0
        fef=PatternFeatureEngineer(); fef.feature_columns=list(X.columns); fef.fit_scaler(Xtr)
        Xtrs=fef.transform(Xtr); Xtes=fef.transform(Xte)
        mdl=_build_xgb(spw); mdl.fit(Xtrs,ytr,eval_set=[(Xtes,yte)],verbose=False)
        yp=mdl.predict(Xtes)
        acc=accuracy_score(yte,yp); pr=precision_score(yte,yp,zero_division=0); rc=recall_score(yte,yp,zero_division=0)
        df_fold=df_tgt.iloc[tei].copy()
        mbt=Backtester(require_valid=use_valid,min_pattern_quality=min_pq,
                       confidence_threshold=cfg.get('confidence_threshold',0.70),
                       atr_sl_mult=cfg.get('atr_sl_mult',1.5),
                       atr_tp_mult=cfg.get('atr_tp_mult',3.0)).run(df_fold,mdl,fef,fef.feature_columns,cfg,use_fold=True)
        fst=str(df_tgt['datetime'].iloc[test_start]) if 'datetime' in df_tgt.columns else ''
        fet=(str(df_tgt['datetime'].iloc[test_end-1]) if test_end<=len(df_tgt) and 'datetime' in df_tgt.columns
             else str(df_tgt['datetime'].iloc[-1]) if 'datetime' in df_tgt.columns else '')
        logging.info(f"  Fold {fold+1}/{n_splits} ({fst[:10]}→{fet[:10]}) Acc:{acc:.2%} Pr:{pr:.2%} Rc:{rc:.2%}")
        if mbt: logging.info(f"    BT|T:{mbt['total_trades']} WR:{mbt['win_rate']}% PF:{mbt['profit_factor']} R:{mbt['total_return_pct']}% DD:{mbt['max_drawdown_pct']}%")
        fold_results.append({'fold':fold+1,'train_size':train_end,'test_size':len(tei),'purge_size':purge_size,
                              'period_start':fst,'period_end':fet,'accuracy':round(acc,4),
                              'precision':round(pr,4),'recall':round(rc,4),'backtest':mbt})
        sc=(mbt.get('profit_factor',1.0)*np.log1p(mbt['total_trades'])+acc if mbt and mbt.get('total_trades',0)>=1 else acc)
        if sc>best_score: best_score=sc; best_model=mdl; best_fe=fef

    if best_model is None:
        if mdl is not None: logging.warning("Fallback: آخر fold"); best_model=mdl; best_fe=fef
        else: raise ValueError("Walk-forward فشل: كل الـ folds تم تخطيها")
    return fold_results,best_model,best_fe


# ══════════════════════════════════════════════════════════════
# TRAIN SINGLE TIMEFRAME
# ══════════════════════════════════════════════════════════════
def train_pattern_model(df_raw, tf, n_splits=6):
    cfg=TIMEFRAME_CONFIG[tf]; min_candles=cfg.get('min_candles',300)
    horizon=cfg.get('horizon',5); min_return=cfg.get('min_return',0.003)
    logging.info("="*60); logging.info(f"فريم: {tf} | v3.4_classical (مع إدارة مخاطر وثقة)"); logging.info("="*60)

    df_tf=(df_raw.set_index('datetime').resample(cfg['resample_rule'])
           .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'}).dropna().reset_index())
    if 'datetime' not in df_tf.columns: df_tf=df_tf.rename(columns={df_tf.columns[0]:'datetime'})
    if cfg.get('years'):
        cut=df_tf['datetime'].max()-timedelta(days=365*cfg['years']); df_tf=df_tf[df_tf['datetime']>=cut].copy()
    logging.info(f"شموع: {len(df_tf):,} | {df_tf['datetime'].min().date()} → {df_tf['datetime'].max().date()}")
    if len(df_tf)<min_candles: logging.error(f"بيانات غير كافية ({len(df_tf)}<{min_candles})"); return None

    fe=PatternFeatureEngineer()
    df_feat=fe.create_features(df_tf,config=cfg,shift=True)
    df_tgt=fe.create_target(df_feat,horizon=horizon,min_return=min_return)
    if df_tgt.empty or len(df_tgt)<max(min_candles//2,30): logging.error("بيانات غير كافية بعد target"); return None

    pos_ratio=df_tgt['target'].mean()
    if pos_ratio<0.05 or pos_ratio>0.95:
        logging.warning(f"target غير متوازن ({pos_ratio:.1%}) — fallback")
        df_tgt2=df_feat.copy()
        df_tgt2['future_close']=df_tgt2['close'].shift(-horizon)
        df_tgt2['pct_change']=(df_tgt2['future_close']-df_tgt2['close'])/(df_tgt2['close']+1e-9)
        df_tgt2['target']=(df_tgt2['pct_change']>df_tgt2['pct_change'].median()).astype(int)
        df_tgt2.dropna(subset=['future_close'],inplace=True); df_tgt=df_tgt2

    df_tgt=df_tgt.reset_index(drop=True); X=df_tgt[fe.feature_columns]; y=df_tgt['target']
    logging.info(f"X.shape={X.shape} | positives={y.mean():.1%}")

    stats={}
    for pc in ['hs_pattern','hs_valid','double_pattern','double_valid','triple_pattern','triple_valid',
               'triangle_pattern','wedge_pattern','flag_pattern','cup_handle_pattern','cup_handle_valid',
               'rectangle_pattern','harmonic_pattern','harmonic_valid','elliott_wave3','elliott_extended_w3',
               'elliott_impulse','elliott_corrective','elliott_valid','pattern_cluster','strong_cluster','pattern_valid']:
        if pc in df_tgt.columns: stats[pc]=int((df_tgt[pc]>0).sum())
    logging.info(f"Pattern stats: {stats}")

    n_folds=4 if tf=="1W" else n_splits
    try:
        fold_res,best_model,best_fe=walk_forward_expanding(X,y,df_tgt,cfg,n_splits=n_folds,purge_pct=0.01,tf_name=tf)
    except ValueError as e:
        logging.error(f"Walk-Forward فشل: {e}"); return None
    if not fold_res: logging.error(f"لا توجد folds — {tf}"); return None

    wf_accs=[f['accuracy'] for f in fold_res]; wf_prs=[f['precision'] for f in fold_res]
    wf_pfs=[f['backtest']['profit_factor'] for f in fold_res if f.get('backtest') and f['backtest'].get('profit_factor')]
    wf_wrs=[f['backtest']['win_rate'] for f in fold_res if f.get('backtest') and f['backtest'].get('win_rate')]
    if wf_accs: logging.info(f"  Avg Acc: {np.mean(wf_accs):.2%} ± {np.std(wf_accs):.2%}")
    if wf_pfs:  logging.info(f"  Avg PF: {np.mean(wf_pfs):.2f} | Consistency: {sum(1 for p in wf_pfs if p>1)}/{len(wf_pfs)}")

    fef_full=PatternFeatureEngineer(); fef_full.fit_scaler(X); Xfs=fef_full.transform(X)
    neg_f=(y==0).sum(); pos_f=(y==1).sum(); best_est=getattr(best_model,'n_estimators',1000)
    final=xgb.XGBClassifier(objective='binary:logistic',n_estimators=best_est,
        max_depth=best_model.max_depth,learning_rate=best_model.learning_rate,
        subsample=best_model.subsample,colsample_bytree=best_model.colsample_bytree,
        reg_alpha=best_model.reg_alpha,reg_lambda=best_model.reg_lambda,
        min_child_weight=best_model.min_child_weight,gamma=best_model.gamma,
        scale_pos_weight=neg_f/pos_f if pos_f else 1,
        random_state=42,tree_method='hist',verbosity=0,use_label_encoder=False,eval_metric='logloss')
    final.fit(Xfs,y)

    feat_imp=pd.DataFrame({'feature':fef_full.feature_columns,'importance':final.feature_importances_}).sort_values('importance',ascending=False).head(20)
    logging.info("أهم 20 ميزة:\n"+feat_imp.to_string(index=False))

    hs=int(len(df_tgt)*0.85); use_valid=tf in('4h','1d','1W','2h')
    min_pq=cfg.get('min_pattern_quality',0.30)
    hmbt=Backtester(require_valid=use_valid,min_pattern_quality=min_pq,
                    confidence_threshold=cfg.get('confidence_threshold',0.70),
                    atr_sl_mult=cfg.get('atr_sl_mult',1.5),
                    atr_tp_mult=cfg.get('atr_tp_mult',3.0)).run(
        df_tgt.iloc[hs:].copy(),final,fef_full,fef_full.feature_columns,cfg,use_fold=False)
    if hmbt: logging.info(f"✅ Holdout|T:{hmbt['total_trades']} WR:{hmbt['win_rate']}% PF:{hmbt['profit_factor']} R:{hmbt['total_return_pct']}% DD:{hmbt['max_drawdown_pct']}%")

    os.makedirs(MODELS_DIR,exist_ok=True)
    mpath=os.path.join(MODELS_DIR,f"{SYMBOL_SAFE}_{tf}_patterns.pkl")
    joblib.dump({'model':final,'scaler':fef_full.scaler,'feature_columns':fef_full.feature_columns,
                 'config':cfg,'feature_importance':feat_imp.to_dict('records')},mpath)
    jpath=os.path.join(MODELS_DIR,f"{SYMBOL_SAFE}_{tf}_patterns_metrics.json")
    with open(jpath,'w') as f:
        json.dump({'version':'v3.4_classical','timeframe':tf,
                   'walk_forward':{'n_folds':len(fold_res),
                       'avg_accuracy':round(float(np.mean(wf_accs)),4) if wf_accs else 0,
                       'std_accuracy':round(float(np.std(wf_accs)),4) if wf_accs else 0,
                       'avg_precision':round(float(np.mean(wf_prs)),4) if wf_prs else 0,
                       'avg_win_rate':round(float(np.mean(wf_wrs)),2) if wf_wrs else 0,
                       'avg_profit_factor':round(float(np.mean(wf_pfs)),2) if wf_pfs else 0,
                       'consistency_pct':round(sum(1 for p in wf_pfs if p>1)/max(len(wf_pfs),1)*100,1),
                       'folds':fold_res},
                   'holdout_last_15pct':hmbt,'config':cfg,'pattern_stats':stats,
                   'training_samples':len(df_tgt),'features':len(fef_full.feature_columns),
                   'feature_importance':feat_imp.to_dict('records')},f,indent=2,default=str)
    logging.info(f"✅ محفوظ: {mpath}")
    return final,fef_full


# ══════════════════════════════════════════════════════════════
# TRAIN ALL
# ══════════════════════════════════════════════════════════════
def train_all_pattern_models(force_refresh_data=False):
    logging.info("="*70)
    logging.info("train_patterns.py  v3.4_classical")
    logging.info("Classic: H&S, Double, Triple, Triangle, Wedge, Flag, Channel, Cup, Rectangle")
    logging.info("Harmonic: Gartley, Butterfly, Bat, Crab, Shark, Cypher  +  PRZ Shifted")
    logging.info("Elliott: Impulse (Strict), Corrective A-B-C, Extended W3, Alternation+W5=W1")
    logging.info("Zigzag: Non-Repainting | vol_ma/dist_SR: Shifted | PRZ: Shifted")
    logging.info("Risk Management: SL/TP via ATR, Confidence Threshold >0.7 for trading")
    logging.info(f"Output: {MODELS_DIR}/")
    logging.info("="*70)
    df_raw=get_or_update_data(force_refresh=force_refresh_data)
    if df_raw.empty: logging.error("فشل تحميل الداتا"); return
    logging.info(f"الداتا: {len(df_raw):,} شمعة | {df_raw['datetime'].min()} → {df_raw['datetime'].max()}")
    results={}; wf_summary={}
    for tf in TIMEFRAME_CONFIG:
        logging.info(f"\n{'─'*50}\nفريم: {tf}")
        res=train_pattern_model(df_raw,tf,n_splits=6); results[tf]="✅" if res else "❌"
        jpath=os.path.join(MODELS_DIR,f"{SYMBOL_SAFE}_{tf}_patterns_metrics.json")
        if os.path.exists(jpath):
            with open(jpath) as f:
                m=json.load(f); wf=m.get('walk_forward',{})
                wf_summary[tf]={'folds':wf.get('n_folds','-'),'avg_acc':wf.get('avg_accuracy','-'),
                                 'avg_pf':wf.get('avg_profit_factor','-'),'consistency':wf.get('consistency_pct','-')}
    logging.info("\n"+"="*70); logging.info("الملخص النهائي:")
    logging.info(f"  {'TF':<8} {'Status':<6} {'Folds':<7} {'Avg Acc':<10} {'Avg PF':<9} {'Consist%'}")
    for tf,st in results.items():
        wf=wf_summary.get(tf,{})
        logging.info(f"  {tf:<8} {st:<6} {str(wf.get('folds','-')):<7} {str(wf.get('avg_acc','-')):<10} {str(wf.get('avg_pf','-')):<9} {str(wf.get('consistency','-'))}%")
    logging.info(f"\nجميع الملفات في: {MODELS_DIR}/"); logging.info("="*70)


if __name__ == "__main__":
    train_all_pattern_models(force_refresh_data=False)


In [13]:
# ============================================================
# download_models


import os
import shutil
from google.colab import files

MODELS_DIR = "models/patterns"
ZIP_NAME   = "XAUUSD_pattern_models.zip"

# ── تحقق من وجود الفولدر ──
if not os.path.exists(MODELS_DIR):
    print(f"❌ الفولدر '{MODELS_DIR}' مش موجود!")
    print("تأكد إن train_patterns_v35.py اشتغل وحفظ الملفات أولاً.")
else:
    # ── اعرض الملفات الموجودة ──
    all_files = os.listdir(MODELS_DIR)
    pkl_files  = [f for f in all_files if f.endswith('.pkl')]
    json_files = [f for f in all_files if f.endswith('.json')]

    print(f"📁 محتوى '{MODELS_DIR}':")
    print(f"   .pkl  (موديلات): {len(pkl_files)}")
    print(f"   .json (مقاييس):  {len(json_files)}")
    print()
    for f in sorted(all_files):
        size = os.path.getsize(os.path.join(MODELS_DIR, f))
        print(f"   {f:<45} {size/1024:.1f} KB")

    # ── اضغط الفولدر ──
    print(f"\n📦 جاري ضغط الفولدر ...")
    shutil.make_archive(
        ZIP_NAME.replace('.zip', ''),  # اسم الملف
        'zip',                          # نوع الضغط
        '.',                            # المجلد الأساسي
        MODELS_DIR                      # الفولدر المراد ضغطه
    )

    size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
    print(f"✅ تم إنشاء: {ZIP_NAME} ({size_mb:.1f} MB)")

    # ── نزّل الملف ──
    print(f"⬇️  جاري التحميل ...")
    files.download(ZIP_NAME)
    print(f"✅ تم! افتح الـ ZIP وهتلاقي فولدر 'models/patterns' بداخله.")

📁 محتوى 'models/patterns':
   .pkl  (موديلات): 8
   .json (مقاييس):  8

   XAUUSD_15min_patterns.pkl                     3797.8 KB
   XAUUSD_15min_patterns_metrics.json            5.1 KB
   XAUUSD_1W_patterns.pkl                        943.4 KB
   XAUUSD_1W_patterns_metrics.json               7.6 KB
   XAUUSD_1d_patterns.pkl                        1652.5 KB
   XAUUSD_1d_patterns_metrics.json               6.1 KB
   XAUUSD_1h_patterns.pkl                        3436.5 KB
   XAUUSD_1h_patterns_metrics.json               5.1 KB
   XAUUSD_2h_patterns.pkl                        3058.3 KB
   XAUUSD_2h_patterns_metrics.json               6.0 KB
   XAUUSD_45min_patterns.pkl                     3379.5 KB
   XAUUSD_45min_patterns_metrics.json            6.0 KB
   XAUUSD_4h_patterns.pkl                        2706.0 KB
   XAUUSD_4h_patterns_metrics.json               5.1 KB
   XAUUSD_5min_patterns.pkl                      3876.2 KB
   XAUUSD_5min_patterns_metrics.json             5.1 KB

📦 جاري ض

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ تم! افتح الـ ZIP وهتلاقي فولدر 'models/patterns' بداخله.
